In [1]:
# Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import os

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.1)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Load Data
print("Loading data...")
df_docs = pd.read_csv("data/documents.csv")
df_queries = pd.read_csv("data/queries.csv")

df_docs['ID'] = df_docs['ID'].astype(str)
df_queries['ID'] = df_queries['ID'].astype(str)

print(f"Loaded {len(df_docs)} documents and {len(df_queries)} queries.")

Loading data...
Loaded 18316 documents and 10 queries.


In [3]:
# Load Model & Generate Embeddings
model_name = 'all-MiniLM-L6-v2'
print(f"Loading Model {model_name}...")
model = SentenceTransformer(model_name)

print("Vectorizing Documents...")
doc_embeddings = model.encode(df_docs['Text'].tolist(), show_progress_bar=True)

print("Vectorizing Queries...")
query_embeddings = model.encode(df_queries['Text'].tolist(), show_progress_bar=True)

print("Saving embeddings to disk...")
np.save("doc_embeddings.npy", doc_embeddings)
np.save("query_embeddings.npy", query_embeddings)
np.save("doc_ids.npy", df_docs['ID'].values)
np.save("query_ids.npy", df_queries['ID'].values)

Loading Model all-MiniLM-L6-v2...
Vectorizing Documents...


Batches: 100%|██████████| 573/573 [01:52<00:00,  5.08it/s]


Vectorizing Queries...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.31it/s]


Saving embeddings to disk...


In [1]:
import numpy as np
import faiss
import os

In [2]:
print("Preparing FAISS...")
doc_embeddings = np.load("doc_embeddings.npy")
query_embeddings = np.load("query_embeddings.npy")
doc_ids = np.load("doc_ids.npy", allow_pickle=True)
query_ids = np.load("query_ids.npy", allow_pickle=True)

doc_embeddings = doc_embeddings.astype('float32')
query_embeddings = query_embeddings.astype('float32')

print("Normalizing vectors for Cosine Similarity...")
faiss.normalize_L2(doc_embeddings)
faiss.normalize_L2(query_embeddings)

Preparing FAISS...
Normalizing vectors for Cosine Similarity...


In [3]:
# Build FAISS Index
print("Building FAISS Index...")
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"Index ready with {index.ntotal} documents.")

Building FAISS Index...
Index ready with 18316 documents.


In [4]:
# Search
k = 50
print(f"Searching for {len(query_embeddings)} queries...")
distances, indices = index.search(query_embeddings, k)

Searching for 10 queries...


In [5]:
# Save Results
results_dir = "results"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

output_file = f"{results_dir}/phase2_results.txt"
run_name = "Semantic_FAISS"

print(f"Writing results to {output_file}...")
with open(output_file, 'w') as f:
    for q_idx in range(len(query_ids)):
        q_id = query_ids[q_idx]
        
        for rank, doc_idx in enumerate(indices[q_idx]):
            real_doc_id = doc_ids[doc_idx]
            score = distances[q_idx][rank]
            
            f.write(f"{q_id} Q0 {real_doc_id} {rank+1} {score:.4f} {run_name}\n")
            
print(f"Done! Results saved to {output_file}")

Writing results to results/phase2_results.txt...
Done! Results saved to results/phase2_results.txt
